In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
from SDRUtils.data.builder import SDRDataBuilder

cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 17, 30))

sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df[(df["Package indicator"] == True) & (df["UPI Underlier Name"] == "USD-SOFR-COMPOUND")]

CACHE HIT...: 100%|██████████| 1/1 [00:00<00:00, 26.06it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
125,1570208068000000401,NaN,NEWT,TRAD,2025-12-29 12:12:48+00:00,None,IR,None,I,True,...,,NaN,-0.00368,,3.0,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
401,1570348761000001101,NaN,NEWT,TRAD,2025-12-29 12:31:29+00:00,None,IR,None,I,True,...,,NaN,0.0004411,,3.0,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
402,1570348764000001401,NaN,NEWT,TRAD,2025-12-29 12:31:29+00:00,None,IR,None,I,True,...,,NaN,0.0004411,,3.0,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
458,1570391040000000301,NaN,NEWT,TRAD,2025-12-29 12:40:05+00:00,None,IR,None,I,True,...,,NaN,0.446,,3.0,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
460,1570391041000000401,NaN,NEWT,TRAD,2025-12-29 12:40:05+00:00,None,IR,None,I,True,...,,NaN,0.446,,3.0,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7115,1573577765000000201,NaN,NEWT,TRAD,2025-12-29 21:45:15+00:00,None,IR,None,I,True,...,,NaN,-0.0062497,,3.0,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
7125,1573612074000000101,NaN,NEWT,TRAD,2025-12-29 21:47:28+00:00,None,IR,None,I,True,...,,NaN,-0.0062452,,3.0,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
7137,1573633133000000201,NaN,NEWT,TRAD,2025-12-29 21:50:41+00:00,None,IR,None,I,True,...,,3.0,,,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
7139,1573633135000000401,NaN,NEWT,TRAD,2025-12-29 21:50:41+00:00,None,IR,None,I,True,...,,3.0,,,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND


In [3]:
from SDRUtils.products import USD_SOFR_SwapProduct

product = USD_SOFR_SwapProduct()
cdf = product.build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True)
cdf

FETCHING DELIVERY BASKETS...: 100%|██████████| 6/6 [00:01<00:00,  3.41it/s]


,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,is_mac,risk
0,1570142946000000101,2025-12-29 12:03:13+00:00,2026-08-06,2036-08-06 00:00:00,OIS_SWAP,10.147222,10Y,True,0.611111,7M,...,None,False,NaN,NaN,NaN,2036-08-06,NaN,NaN,False,2500.0
1,1570299802000000201,2025-12-29 12:05:58+00:00,2026-03-18,2046-03-18 00:00:00,OIS_SWAP,20.294444,IMM_H2046,True,0.219444,IMM_H2026,...,None,False,NaN,NaN,NaN,2046-03-18,NaN,NaN,False,1400.0
2,1570469108000000701,2025-12-29 12:09:34+00:00,2026-03-18,2051-03-18 00:00:00,OIS_SWAP,25.369444,IMM_H2051,True,0.219444,IMM_H2026,...,None,False,NaN,NaN,NaN,2051-03-18,NaN,NaN,False,1600.0
3,1570184451000000301,2025-12-29 12:09:46+00:00,2026-03-18,2031-03-18 00:00:00,OIS_SWAP,5.072222,IMM_H2031,True,0.219444,IMM_H2026,...,None,False,NaN,NaN,NaN,2031-03-18,NaN,NaN,False,18200.0
4,1570196242000000201,2025-12-29 12:11:05+00:00,2026-03-18,2036-03-18 00:00:00,OIS_SWAP,10.147222,IMM_H2036,True,0.219444,IMM_H2026,...,None,False,NaN,NaN,NaN,2036-03-18,NaN,NaN,False,9100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1605,1573649665000000201,2025-12-29 21:53:50+00:00,2025-12-31,2029-12-31 00:00:00,OIS_SWAP,4.058333,4Y,False,0.005556,spot,...,[1573649665000000201],True,91282CMD0,5-Year,2024-12-31,2029-12-31,low,NaN,False,1500.0
1606,1573656731000000101,2025-12-29 21:54:42+00:00,2025-12-31,2026-12-31 00:00:00,OIS_SWAP,1.013889,1Y,False,0.005556,spot,...,[1573656731000000101],True,91282CME8,2-Year,2024-12-31,2026-12-31,low,NaN,False,4000.0
1607,1573691775000000101,2025-12-29 21:59:26+00:00,2025-12-31,2032-12-31 00:00:00,OIS_SWAP,7.102778,7Y,False,0.005556,spot,...,[1573691775000000101],True,91282CPQ8,7-Year,2025-12-31,2032-12-31,low,NaN,False,2500.0
1608,1573723141000000201,2025-12-29 21:52:38+00:00,2025-12-31,2055-11-15 00:00:00,OIS_SWAP,30.308333,30Y,False,0.005556,spot,...,[1573723141000000201],True,912810UP1,30-Year,2025-11-17,2055-11-15,high,NaN,False,100100.0


In [10]:
# cdf[cdf["trade_id"] == 1573577765000000201].iloc[0].to_dict(), cdf[cdf["trade_id"] == 1573612074000000101].iloc[0].to_dict()

cdf[(cdf["package_legs"].isna()) & (cdf["Package indicator"] == True) & (cdf["Package transaction spread"].notna()) & (cdf["forward_label"] == "spot")]

,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,is_mac,risk
8,1570208068000000401,2025-12-29 12:12:48+00:00,2025-12-31,2035-12-31 00:00:00,OIS_SWAP,10.144444,10Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2035-12-31,NaN,NaN,False,4200.0
12,1570303270000000301,2025-12-29 12:23:58+00:00,2025-12-31,2035-12-31 00:00:00,OIS_SWAP,10.144444,10Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2035-12-31,NaN,NaN,False,50300.0
20,1570397054000000101,2025-12-29 12:40:16+00:00,2025-12-31,2055-12-31 00:00:00,OIS_SWAP,30.436111,30Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2055-12-31,NaN,NaN,False,43200.0
21,1570404072000000101,2025-12-29 12:41:29+00:00,2025-12-31,2035-12-31 00:00:00,OIS_SWAP,10.144444,10Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2035-12-31,NaN,NaN,False,50300.0
30,1570446165000000301,2025-12-29 12:46:58+00:00,2025-12-31,2035-12-31 00:00:00,OIS_SWAP,10.144444,10Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2035-12-31,NaN,NaN,False,62800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,1573625078000000101,2025-12-29 21:49:16+00:00,2025-12-31,2055-12-31 00:00:00,OIS_SWAP,30.436111,30Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2055-12-31,NaN,NaN,False,43200.0
927,1573648099000000201,2025-12-29 21:53:07+00:00,2025-12-31,2055-12-31 00:00:00,OIS_SWAP,30.436111,30Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2055-12-31,NaN,NaN,False,43200.0
928,1573649664000000101,2025-12-29 21:53:07+00:00,2025-12-31,2055-12-31 00:00:00,OIS_SWAP,30.436111,30Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2055-12-31,NaN,NaN,False,43200.0
929,1573660614000000101,2025-12-29 21:55:34+00:00,2025-12-31,2055-12-31 00:00:00,OIS_SWAP,30.436111,30Y,False,0.005556,spot,...,None,False,NaN,NaN,NaN,2055-12-31,NaN,NaN,False,43200.0


In [11]:
df[df["Dissemination Identifier"] == 1570446165000000301].iloc[0].to_dict()

{'Dissemination Identifier': 1570446165000000301,
 'Original Dissemination Identifier': nan,
 'Action type': 'NEWT',
 'Event type': 'TRAD',
 'Event timestamp': Timestamp('2025-12-29 12:46:58+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'I',
 'Mandatory clearing indicator': True,
 'Execution Timestamp': Timestamp('2025-12-29 12:46:58+0000', tz='UTC'),
 'Effective Date': Timestamp('2025-12-31 00:00:00'),
 'Expiration Date': Timestamp('2035-12-31 00:00:00'),
 'Maturity date of the underlier': None,
 'Non-standardized term indicator': False,
 'Platform identifier': 'ISWV',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': None,
 'Notional amount-Leg 1': '75,000,000',
 'Notional amount-Leg 2': '75,000,000',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': 'USD',
 'Notional quantity-Leg 1': None,
 'Notional quantity-Leg 2':